# 🗄️ Database Verification

This notebook verifies the SQLAlchemy database models (`ParentProfile`, `HealthLog`, `ScheduledReminder`) and their CRUD operations using a local SQLite database.

In [1]:
import asyncio
import os
import json
from datetime import datetime, time, timedelta

# Set up the environment to use a local sqlite database instead of Postgres
os.environ["DATABASE_URL"] = "sqlite+aiosqlite:///./test_checkup.db"

from sqlalchemy.ext.asyncio import create_async_engine, async_sessionmaker
from sqlalchemy import select

# Import our models
from checkup.scheduler.models import Base, ParentProfile, HealthLog, ScheduledReminder
from checkup.config import settings

print(f"Testing against: {settings.database_url}")

Testing against: sqlite+aiosqlite:///./test_checkup.db


### 1. Create the Database Tables
We'll generate the tables based on our SQLAlchemy models.

In [3]:
!pip install aiosqlite

  Using cached aiosqlite-0.22.1-py3-none-any.whl.metadata (4.3 kB)
Using cached aiosqlite-0.22.1-py3-none-any.whl (17 kB)


In [7]:
# Create engine and session maker
engine = create_async_engine(settings.database_url, echo=False)
async_session = async_sessionmaker(engine, expire_on_commit=False)

async def setup_db():
    async with engine.begin() as conn:
        # Drop all tables first for a clean run
        await conn.run_sync(Base.metadata.drop_all)
        # Create all tables
        await conn.run_sync(Base.metadata.create_all)
    print("✅ Recreated all database tables.")

await setup_db()

✅ Recreated all database tables.


### 2. Create a Test Parent Profile

In [8]:
async def create_test_parent():
    async with async_session() as session:
        new_parent = ParentProfile(
            parent_phone="919876543210",
            caregiver_phone="+911234567890",
            parent_name="Amma",
            age=68,
            known_conditions=["Knee pain", "hypertension"],
            medications=[
                {"name": "Metformin", "dosage": "500mg", "times": ["08:00", "20:00"]},
                {"name": "Atorvastatin", "dosage": "10mg", "times": ["21:00"]}
            ],
            preferred_language="te",
            checkin_time=time(9, 0) # 9:00 AM daily checkin
        )
        session.add(new_parent)
        await session.commit()
        print(f"✅ Created test parent: {new_parent.parent_name} (ID: {new_parent.id})")
        return new_parent.id

parent_id = await create_test_parent()

✅ Created test parent: Amma (ID: 1)


### 3. Read Profile Back and Verify

In [9]:
async def get_parent(pid: int):
    async with async_session() as session:
        result = await session.execute(select(ParentProfile).where(ParentProfile.id == pid))
        parent = result.scalar_one_or_none()
        
        if parent:
            print(f"Found Parent: {parent.parent_name} (Age {parent.age})")
            print(f"  Phone: {parent.parent_phone} | Caregiver: {parent.caregiver_phone}")
            print(f"  Conditions: {', '.join(parent.known_conditions)}")
            print("  Medications:")
            for med in parent.medications:
                print(f"    - {med['name']} ({med['dosage']}) at {', '.join(med['times'])}")
        else:
            print("❌ Parent not found.")

await get_parent(parent_id)

Found Parent: Amma (Age 68)
  Phone: 919876543210 | Caregiver: +911234567890
  Conditions: Knee pain, hypertension
  Medications:
    - Metformin (500mg) at 08:00, 20:00
    - Atorvastatin (10mg) at 21:00


### 4. Insert Health Logs
Simulating the LangGraph agent storing responses after a daily check-in.

In [10]:
async def add_health_logs(pid: int):
    async with async_session() as session:
        # Good checkin from yesterday
        log1 = HealthLog(
            parent_id=pid,
            timestamp=datetime.utcnow() - timedelta(days=1),
            log_type="checkin",
            data={"overall_feeling": "good", "activity_level": "walk", "pain_level": "none"},
            risk_level="low"
        )
        # Need attention checkin from today
        log2 = HealthLog(
            parent_id=pid,
            timestamp=datetime.utcnow(),
            log_type="checkin",
            data={"overall_feeling": "tired", "pain_level": "knee ache"},
            risk_level="medium"
        )
        session.add_all([log1, log2])
        await session.commit()
        print(f"✅ Added {2} health logs for parent {pid}")

await add_health_logs(parent_id)

✅ Added 2 health logs for parent 1


/var/folders/tr/nb04ltxd3fz30s728x0w3l240000gn/T/ipykernel_82715/2173097202.py:6: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp=datetime.utcnow() - timedelta(days=1),
/var/folders/tr/nb04ltxd3fz30s728x0w3l240000gn/T/ipykernel_82715/2173097202.py:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp=datetime.utcnow(),


### 5. Fetch Health Logs with Relationships

In [11]:
from sqlalchemy.orm import selectinload

async def fetch_profile_with_logs(pid: int):
    async with async_session() as session:
        # Load parent along with all their linked health logs
        query = select(ParentProfile).options(selectinload(ParentProfile.health_logs)).where(ParentProfile.id == pid)
        result = await session.execute(query)
        parent = result.scalar_one()
        
        print(f"Parent: {parent.parent_name} has {len(parent.health_logs)} health logs:")
        for log in sorted(parent.health_logs, key=lambda l: l.timestamp):
            print(f"  [{log.timestamp.strftime('%Y-%m-%d %H:%M')}] Type: {log.log_type.upper()} | Risk: {log.risk_level.upper()} | Data: {log.data}")

await fetch_profile_with_logs(parent_id)

Parent: Amma has 2 health logs:
  [2026-03-01 21:55] Type: CHECKIN | Risk: LOW | Data: {'overall_feeling': 'good', 'activity_level': 'walk', 'pain_level': 'none'}
  [2026-03-02 21:55] Type: CHECKIN | Risk: MEDIUM | Data: {'overall_feeling': 'tired', 'pain_level': 'knee ache'}


### 6. Scheduled Reminders

In [12]:
async def test_reminders(pid: int):
    async with async_session() as session:
        # Add a reminder that was sent, but not acknowledged
        reminder = ScheduledReminder(
            parent_id=pid,
            reminder_type="medication",
            scheduled_time=datetime.utcnow() - timedelta(minutes=30),
            sent=True,
            acknowledged=False
        )
        session.add(reminder)
        await session.commit()
        
        # Mark it as acknowledged (simulating the reply via Whatsapp)
        reminder.acknowledged = True
        await session.commit()
        print(f"✅ Reminder updated successfully! Sent: {reminder.sent}, Acknowledged: {reminder.acknowledged}")

await test_reminders(parent_id)

✅ Reminder updated successfully! Sent: True, Acknowledged: True


/var/folders/tr/nb04ltxd3fz30s728x0w3l240000gn/T/ipykernel_82715/3423793090.py:7: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  scheduled_time=datetime.utcnow() - timedelta(minutes=30),


All Database operations working exactly as expected locally ✅